In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import duckdb
import pandas as pd
from src.possession_parser import Possession_parser

In [2]:
con = duckdb.connect(database='data/nba.sqlite', read_only=False)
# df = con.query("show tables").fetchdf()
# df

In [ ]:
## get game-ids

# df = con.query("select * from play_by_play limit 2").fetchdf()
# df

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,...,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600012,0,12,0,1,14:43 PM,12:00,None,Start of 1st Period (14:43 PM EST),None,...,None,None,0.0,0,None,None,None,None,None,0
1,0029600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None,...,Suns,PHX,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0


In [ ]:
# df = con.query("select * from play_by_play where game_id = '0029600012'").fetchdf()

In [3]:
game_ids_df = con.query("select distinct game_id from play_by_play").fetchdf()
game_ids = game_ids_df['game_id']

In [4]:
cur_game_id = game_ids[0]
game_df = con.query(f"select * from play_by_play where game_id = '{game_ids[0]}'").fetchdf()
game_df = game_df.sort_values(by='eventnum')
game_df

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,...,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600064,1,10,0,1,11:40 PM,12:00,Jump Ball Geiger vs. Baker: Tip to Robinson,None,None,...,Bucks,MIL,5.0,299,Glenn Robinson,1610612749.0,Milwaukee,Bucks,MIL,0
1,0029600064,2,2,5,1,11:41 PM,11:45,None,None,MISS Baker Layup,...,None,None,0.0,0,None,None,None,None,None,0
2,0029600064,3,4,0,1,11:41 PM,11:45,None,None,Bucks Rebound,...,None,None,0.0,0,None,None,None,None,None,0
3,0029600064,4,2,5,1,11:41 PM,11:41,None,None,MISS Allen Layup,...,None,None,0.0,0,None,None,None,None,None,0
4,0029600064,5,4,0,1,11:41 PM,11:40,Mason REBOUND (Off:0 Def:1),None,None,...,None,None,0.0,0,None,None,None,None,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,0029600064,444,5,1,4,13:59 PM,0:08,Geiger STEAL (1 STL),None,Robinson Bad Pass Turnover (P4.T12),...,Hornets,CHH,0.0,0,None,None,None,None,None,0
442,0029600064,445,2,1,4,13:59 PM,0:01,MISS Rice 3PT Jump Shot,None,None,...,None,None,0.0,0,None,None,None,None,None,0
443,0029600064,446,4,0,4,13:59 PM,0:00,Mason REBOUND (Off:3 Def:9),None,None,...,None,None,0.0,0,None,None,None,None,None,0
444,0029600064,447,9,1,4,13:59 PM,0:00,HORNETS Timeout: Regular (Full 7 Short 2),None,None,...,None,None,0.0,0,None,None,None,None,None,0


In [13]:
parser = Possession_parser(game_df)
possessions = parser.parse_game()
possessions

it working
player not on offense scored at row 5
player not on offense scored at row 11
player not on offense scored at row 23
player not on offense scored at row 26
player not on offense scored at row 27
player not on offense scored at row 37
player not on offense scored at row 38
player not on offense scored at row 45
player not on offense scored at row 51
player not on offense scored at row 57
player not on offense scored at row 61
player not on offense scored at row 68
player not on offense scored at row 69
player not on offense scored at row 70
player not on offense scored at row 86
player not on offense scored at row 89
player not on offense scored at row 95
player not on offense scored at row 96
player not on offense scored at row 105
player not on offense scored at row 117
player not on offense scored at row 120
player not on offense scored at row 126
player not on offense scored at row 129
player not on offense scored at row 135
player not on offense scored at row 136
player n

,game_id,possession_id,offense_team_id,possession_end_event_type,possession_end_points
0,0029600064,0,761,4.0,0
1,0029600064,1,452,4.0,0
2,0029600064,2,761,1.5,2
3,0029600064,3,452,4.0,0
4,0029600064,4,761,4.0,0
...,...,...,...,...,...
185,0029600064,185,761,4.0,0
186,0029600064,186,761,5.1,0
187,0029600064,187,452,5.1,0
188,0029600064,188,761,4.0,0


In [ ]:
cur_period = 1
possession_index = 0
game_possessions_df = pd.DataFrame(columns=['game_id', 'possession_id', 'offense_team_id', 'possession_end_event_type', 'possession_end_points'])

def add_possession(offense_team_id, end_event_type, end_points):
    cur_possession = {
        'game_id': cur_game_id,
        'possession_id': possession_index,
        'offense_team_id': offense_team_id,
        'possession_end_event_type': end_event_type,
        'possession_end_points': end_points}
    new_row_df = pd.DataFrame(cur_possession)
    game_possessions_df = pd.concat([game_possessions_df, new_row_df], ignore_index=True)

team_ids = game_df['player1_id'].unique()
team0 = team_ids[0]
team1 = team_ids[1]
is_team1_offense = True

def get_offense_id():
    return team1 if is_team1_offense else team0

for index, row in game_df.iterrows():
    match row['eventmsgtype']:
        case 1: # made shot
            points = 3 if (row['eventmsgactiontype'] == 7) else 2
            add_possession(get_offense_id(), f'{row['eventmsgtype']}.{row['eventmsgactiontype']}', points)
            possession_index += 1
            if row['player1_team_id'] != get_offense_id():
                print('player not on offense scored at row', index)
        # case 4: # rebound
        #     print(row['player1_id'])
        case 5: # turnover
            add_possession(get_offense_id(), f'{row['eventmsgtype']}.{row['eventmsgactiontype']}', points)
            possession_index += 1
            is_team1_offense = not is_team1_offense
        case 10: # jump ball
            # TODO: properly resolve the jump ball
            is_team1_offense = True if team1 == row['player3_team_id'] else False
        case 13: # end of period
            add_possession(get_offense_id(), f'{row['eventmsgtype']}.{row['eventmsgactiontype']}', points)
            cur_period += 1
            possession_index += 1

1610612737
87
281
87
221
895
281
934
281
87
1610612737
87
87
363
711
895
934
87
711
87
895
87
1610612755
1610612755
895
695
707
777
446
446
262
262
1096
87
1610612755
1096
1610612755
1610612755
446
120
363
77
934
120
302
262
934
302
281
281
363
934
1610612737
281
302
363
221
934
363
934
1610612755
934
1610612755
1096
221
934
221
1610612755
262
1096
363
1096
711
895
363
1610612737
221
1096
221
1096
221
1610612737
934
221
262
416
777
87
468
711
468
1610612755
1610612737
707
1096
468
695
1610612737
695
971
1610612755
968
695


In [ ]:
game_df

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,...,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag,possession_index
0,0029600310,1,10,0,1,11:39 PM,12:00,Jump Ball Mutombo vs. Williams: Tip to Laettner,None,None,...,PHI,4.0,363,Christian Laettner,1610612737.0,Atlanta,Hawks,ATL,0,0029600310_0
1,0029600310,2,6,2,1,11:40 PM,11:55,None,None,Weatherspoon S.FOUL (P1.T1),...,PHI,0.0,0,None,None,None,None,None,0,0029600310_0
2,0029600310,3,3,11,1,11:40 PM,11:55,MISS Smith Free Throw 1 of 2,None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_0
3,0029600310,4,4,0,1,11:40 PM,11:55,HAWKS Rebound,None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_0
4,0029600310,5,3,12,1,11:40 PM,11:55,Smith Free Throw 2 of 2 (1 PTS),None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432,0029600310,442,1,5,4,13:41 PM,0:16,None,None,Bradtke Layup (4 PTS) (Harris 1 AST),...,PHI,0.0,0,None,None,None,None,None,0,0029600310_3
433,0029600310,443,5,2,4,13:41 PM,0:10,Barry Lost Ball Turnover (P2.T11),None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_3
434,0029600310,445,2,1,4,13:41 PM,0:02,None,None,MISS Davis 25' 3PT Jump Shot,...,None,0.0,0,None,None,None,None,None,0,0029600310_3
435,0029600310,446,4,0,4,13:41 PM,0:00,Recasner REBOUND (Off:0 Def:4),None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_3


In [ ]:
df[["homedescription",	"neutraldescription"	,"visitordescription"]]

,homedescription,neutraldescription,visitordescription
0,None,Start of 1st Period (14:43 PM EST),None
1,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None
2,None,None,MISS Cassell 15' Jump Shot
3,O'Neal REBOUND (Off:0 Def:1),None,None
4,MISS Ceballos 26' 3PT Jump Shot,None,None
...,...,...,...
461,None,None,MISS Manning Free Throw 1 of 2
462,None,None,Suns Rebound
463,None,None,MISS Manning Free Throw 2 of 2
464,Knight REBOUND (Off:0 Def:2),None,None


In [ ]:
df.columns

Index(['game_id', 'eventnum', 'eventmsgtype', 'eventmsgactiontype', 'period',
       'wctimestring', 'pctimestring', 'homedescription', 'neutraldescription',
       'visitordescription', 'score', 'scoremargin', 'person1type',
       'player1_id', 'player1_name', 'player1_team_id', 'player1_team_city',
       'player1_team_nickname', 'player1_team_abbreviation', 'person2type',
       'player2_id', 'player2_name', 'player2_team_id', 'player2_team_city',
       'player2_team_nickname', 'player2_team_abbreviation', 'person3type',
       'player3_id', 'player3_name', 'player3_team_id', 'player3_team_city',
       'player3_team_nickname', 'player3_team_abbreviation',
       'video_available_flag'],
      dtype='object')

In [ ]:

## Pseudo logic for aggregation of player counting stats


# from collections import Counter

# players = Counter()

# for game in games:
#     for event in game.events:
#         player = event.player
#         stat = event.stat
#         players[player[stat]] += 1 

